In [1]:
import json
import requests
from utils import *

In [ ]:
messages = [{
    "role": "user",
    "content": "What was a positive news story from today?"
}]
resp = requests.post("https://api.anthropic.com/v1/messages", headers=ant_headers, json={
    "model": "claude-sonnet-5",
    "max_tokens": 4096,
    "messages": messages,
    "tools": [{
      "type": "web_search_20260318",
      "name": "web_search"
    }]
})
resp

In [ ]:
preview(resp.json()['content'], 50)

In [ ]:
messages.append({
    "role": "assistant",
    "content": resp.json()["content"]
})

In [ ]:
messages.append({
    "role": "user",
    "content": "Which one is your favorite one?"
})

In [ ]:
resp = requests.post("https://api.anthropic.com/v1/messages", headers=ant_headers, json={
    "model": "claude-sonnet-5",
    "max_tokens": 4096,
    "messages": messages,
    # "tools": [{
    #   "type": "web_search_20260318",
    #   "name": "web_search"
    # }]
})
resp

In [ ]:
resp.json()['content']

### Basic

### Non streaming

In [20]:
resp = requests.post("https://api.anthropic.com/v1/messages", headers=ant_headers, json={
    "model": "claude-sonnet-5",
    "max_tokens": 4096,
    "output_config": {
        "effort": "medium"
    },
    "messages": [
      {
        "role": "user",
        "content": "Compare Apple’s and NVIDIA’s stock performance today. State each company’s current price and percentage change, explain one reported reason for each movement, and cite a different source for every factual claim."
      }
    ],
    "tools": [{
      "type": "web_search_20260318",
      "name": "web_search",
      "max_uses": 10,
      "allowed_callers": ["direct"]
    }]
})
resp

<Response [200]>

In [55]:
def preview(obj, max_length=50, key_names=None, whitelist_keys=None):
    key_names = set(key_names or [])
    whitelist_keys = set(whitelist_keys or [])

    def walk(x):
        if isinstance(x, dict):
            
            return {
                k: (
                    (v if isinstance(v, str) and len(v) <= max_length else v[:max_length] + "<redacted for length> ...")
                    if isinstance(v, str) and (k not in whitelist_keys)
                    
                    # (key_names is None or k in key_names)
                    else walk(v)
                )
                for k, v in x.items()
            }

        if isinstance(x, list):
            return [walk(v) for v in x]

        if isinstance(x, tuple):
            return tuple(walk(v) for v in x)

        if isinstance(x, set):
            return {walk(v) for v in x}

        if isinstance(x, frozenset):
            return frozenset(walk(v) for v in x)

        return x

    return walk(obj)

In [57]:
preview(resp.json()['content'][:10], 50)

[{'type': 'thinking',
  'thinking': '',
  'signature': 'EsMCCpABCBAYAipAxxBlI+M26QbX64FNN7A/TjUa8X8ow9zxAA<redacted for length> ...'},
 {'type': 'server_tool_use',
  'id': 'srvtoolu_01V6vsJNQSpfd8xCAiHDo8cd',
  'name': 'web_search',
  'input': {'query': 'Apple stock price today'}},
 {'type': 'server_tool_use',
  'id': 'srvtoolu_017PBNPWYH8G8CwhRzQd8fBk',
  'name': 'web_search',
  'input': {'query': 'NVIDIA stock price today'}},
 {'type': 'web_search_tool_result',
  'tool_use_id': 'srvtoolu_01V6vsJNQSpfd8xCAiHDo8cd',
  'content': [{'type': 'web_search_result',
    'title': '__symbol__ Stock Quote Price and Forecast | CNN',
    'url': 'https://www.cnn.com/markets/stocks/AAPL',
    'encrypted_content': 'Eq0NCioIEhgCIiRmMDQ1ZTA1Yy02MjA3LTQ0YTEtOTcxMi1lY2<redacted for length> ...',
    'page_age': '4 days ago'},
   {'type': 'web_search_result',
    'title': 'Apple Stock Chart — NASDAQ:AAPL Stock Price — Trad<redacted for length> ...',
    'url': 'https://www.tradingview.com/symbols/NASDAQ-A

In [45]:
c

{'type': 'web_search_tool_result',
 'tool_use_id': 'srvtoolu_01V6vsJNQSpfd8xCAiHDo8cd',
 'content': [{'type': 'web_search_result',
   'title': '__symbol__ Stock Quote Price and Forecast | CNN',
   'url': 'https://www.cnn.com/markets/stocks/AAPL',
   'encrypted_content': 'Eq0NCioIEhgCIiRmMDQ1ZTA1Yy02MjA3LTQ0YTEtOTcxMi1lY2I2ZjlmNmEyMzYSDCH03QJ6uTYVIEW4dhoMR8Dfyy39+BGD6x+jIjDz8bEHnhzAu/gprESWjoUe0tuI6/CFAPc+NifR7YHKTEWxKpOgd6ehk9N/XSVupwMqsAyCslrbJABHPW2SWa5T58eZQGywB/XsRmR9VevBrXR/nNJcGRioNmpmx/MyctOS5q30VcD86OUfpDdlPTLJ6Z39mQEuQdwpjKmaD5z7dbJR6m/riyTVgh+cxBvNI1AzI2RfMtOIYRYtgwoKpdO4VdyRHweQdmb45hVNRQxsHI9rhFepUKYG4JyLDXEQVVq4PPMcfuugL0elY6Qhh5mV3nSl3JGXg670R/fbiMdJ6gwc56RvnTXZ8kbQmYoAQzQuPKaImkU1EQ1US6A7UTOIkOHRxGrzoJpu6HWLxzWlSDhofXu5WaxWZ4jCAwiMAR+jxclP/TTKmlKg34SqDR2pxH7f2LnWRIuAlvyfus7gCuFgPatT5KWwg/r3urhz3GN4UHAgBywj7wvZ7dZlSGwDbjTIy/2TTSwIeFx8zDocAwF+81RykJN97kbfNyJ7X49KJXx5SmDFjxEWVcFlE6NU23aDGrRRObSj/pgv+91aX1mt5ZGJQDrmOd0tpeDrSnB1uajqICxYNHOM0cnp1C5hzuORjbAmQKT1ez075gUi++njMOuG

In [ ]:
preview(resp.json(), 50)

In [15]:
with open('anthropic_basic_non_streaming_output_2.json', 'w') as fp:
    json.dump(preview(resp.json()), fp, indent=2)

#### Streaming

In [16]:
resp = requests.post(
    "https://api.anthropic.com/v1/messages",
    headers=ant_headers,
    json={
        "model": "claude-sonnet-5",
        "max_tokens": 4096,
        "output_config": {
        "effort": "medium"
    },
        "messages": [{
            "role": "user",
            "content": "Compare Apple’s and NVIDIA’s stock performance today. State each company’s current price and percentage change, explain one reported reason for each movement, and cite a different source for every factual claim.",
        }],
        "tools": [{
            "type": "web_search_20260318",
            "name": "web_search",
            "max_uses": 10,
      "allowed_callers": ["direct"]
        }],
        "stream": True,
    },
    stream=True,
)

In [17]:
resp

<Response [200]>

In [18]:
lines = list(resp.iter_lines(decode_unicode=True))

In [19]:
with open('anthropic_basic_streaming_lines_2.txt', 'w') as fp:
    for line in lines:
        if not line.strip():
            continue
        if line.startswith("data:"):
            line = json.dumps(preview(json.loads(line[6:]), 50), indent=2) + "\n"
        print(line, file=fp)

## Code Execution caller

From Anthropic docs:

```
With basic web search, every search result is loaded into Claude's context window, and much of that content can be irrelevant to the request. With web_search_20260209 or later, Claude instead writes and runs code that filters the results first, so only relevant content reaches the context window. This reduces token use on search-heavy requests.

Dynamic filtering runs web search from inside code execution: on web_search_20260209 and later, the tool's allowed_callers field defaults to ["code_execution_20260120"], and when dynamic filtering runs, the API provisions the code execution it needs for the request automatically. You don't need to add the code execution tool to tools yourself. There are no additional charges for code execution calls made this way beyond the standard token costs.

To call web search directly, without dynamic filtering, set allowed_callers: ["direct"]. Models that don't support programmatic tool calling require this setting. Without it, the API returns a 400 error that tells you to set it.
```

#### Non Streaming

In [9]:
resp = requests.post("https://api.anthropic.com/v1/messages", headers=ant_headers, json={
    "model": "claude-sonnet-5",
    "max_tokens": 4096,
    "output_config": {
        "effort": "low"
    },
    "messages": [
      {
        "role": "user",
        "content": "What's the price of AAPL today?"
      }
    ],
    "tools": [{
      "type": "web_search_20260318",
      "name": "web_search",
      "max_uses": 5,
      "allowed_callers": ["code_execution_20260120"]
    }]
})
resp

<Response [200]>

In [10]:
with open('anthropic_basic_code_execution_caller_non_streaming_output.json', 'w') as fp:
    json.dump(preview(resp.json()), fp, indent=2)

#### Streaming

In [11]:
resp = requests.post(
    "https://api.anthropic.com/v1/messages",
    headers=ant_headers,
    json={
        "model": "claude-sonnet-5",
        "max_tokens": 4096,
        "output_config": {
        "effort": "low"
    },
        "messages": [{
            "role": "user",
            "content": "What's the price of AAPL today?",
        }],
        "tools": [{
            "type": "web_search_20260318",
            "name": "web_search",
            "max_uses": 5,
            "allowed_callers": ["code_execution_20260120"]
        }],
        "stream": True,
    },
    stream=True,
)
resp

<Response [200]>

In [12]:
lines = list(resp.iter_lines(decode_unicode=True))

In [13]:
with open('anthropic_basic_code_execution_caller_streaming_lines.txt', 'w') as fp:
    for line in lines:
        if not line.strip():
            continue
        if line.startswith("data:"):
            line = json.dumps(preview(json.loads(line[6:]), 50), indent=2) + "\n"
        print(line, file=fp)

---

### Web Fetch

In [ ]:
prompt = """Search the web for Apple's latest quarterly earnings release.

Then open the official Apple Investor Relations page containing that earnings release and inspect the page itself before answering.

From that page, tell me:
1. The reported quarterly revenue
2. The reported diluted EPS
3. The publication date

Do not answer from search-result snippets alone. You must open the source page and read it."""

#### Non streaming

In [ ]:
resp = requests.post("https://api.anthropic.com/v1/messages", headers=ant_headers, json={
    "model": "claude-sonnet-5",
    "max_tokens": 4096,
    "messages": [
      {
        "role": "user",
        "content": prompt
      }
    ],
    "tools": [{
      "type": "web_search_20260318",
      "name": "web_search"
    }, {
      "type": "web_fetch_20260318",
      "name": "web_fetch"
    }]
})
resp

In [ ]:
preview(resp.json()['content'], 50)

In [ ]:
with open('anthropic_web_fetch_non_streaming_output.json', 'w') as fp:
    json.dump(preview(resp.json()), fp, indent=2)

#### Streaming

In [ ]:
resp = requests.post(
    "https://api.anthropic.com/v1/messages",
    headers=ant_headers,
    json={
        "model": "claude-sonnet-5",
        "max_tokens": 4096,
        "messages": [{
            "role": "user",
            "content": prompt
        }],
        "tools": [{
            "type": "web_search_20260318",
            "name": "web_search",
        }, {
          "type": "web_fetch_20260318",
          "name": "web_fetch"
        }],
        "stream": True,
    },
    stream=True,
)

In [ ]:
resp

In [ ]:
lines = list(resp.iter_lines(decode_unicode=True))

In [ ]:
with open('anthropic_web_fetch_streaming_output.txt', 'w') as fp:
    for line in lines:
        if not line.strip():
            continue
        if line.startswith("data:"):
            line = json.dumps(preview(json.loads(line[6:]), 50), indent=2) + "\n"
        print(line, file=fp)

---

### Web fetch only

In [ ]:
resp = requests.post("https://api.anthropic.com/v1/messages", headers=ant_headers, json={
    "model": "claude-sonnet-5",
    "max_tokens": 4096,
    "messages": [
      {
        "role": "user",
        "content": "Summarize this article: https://www.apple.com/newsroom/2026/07/apple-reports-third-quarter-results/"
      }
    ],
    "tools": [{
      "type": "web_fetch_20260318",
      "name": "web_fetch"
    }]
})
resp

In [ ]:
preview(resp.json(), 50)

In [ ]:
with open('anthropic_web_fetch_only_non_streaming_output.json', 'w') as fp:
    json.dump(preview(resp.json()), fp, indent=2)

#### Streaming

In [ ]:
resp = requests.post(
    "https://api.anthropic.com/v1/messages",
    headers=ant_headers,
    json={
        "model": "claude-sonnet-5",
        "max_tokens": 4096,
        "messages": [{
            "role": "user",
            "content": prompt
        }],
        "tools": [{
          "type": "web_fetch_20260318",
          "name": "web_fetch"
        }],
        "stream": True,
    },
    stream=True,
)

In [ ]:
resp

In [ ]:
lines = list(resp.iter_lines(decode_unicode=True))

In [ ]:
with open('anthropic_web_fetch_only_streaming_output.txt', 'w') as fp:
    for line in lines:
        if not line.strip():
            continue
        if line.startswith("data:"):
            line = json.dumps(preview(json.loads(line[6:]), 50), indent=2) + "\n"
        print(line, file=fp)